# LAB 3 — Streaming & Incremental Ingestion (Week 3) (Part 1)

In [0]:
dbutils.widgets.text("catalog", "", "Catalog: ")
dbutils.widgets.text("schema", "", "Schema: ")

In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = "raw_data"

original_file_path = f"/Volumes/{catalog}/{schema}/{volume}/co2.csv"
lab3_source_path = f"/Volumes/{catalog}/{schema}/{volume}/lab3_stream_source/"
lab3_checkpoint = f"/Volumes/{catalog}/{schema}/{volume}/_checkpoints/lab3_co2/"

table_name = f"{catalog}.{schema}.co2_bronze_lab3"

In [0]:
co2_df = (spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(original_file_path)
)

In [0]:
(co2_df.repartition(1000)
    .write
    .mode("overwrite")
    .format("csv")
    .option("header", "true")
    .save(lab3_source_path)
)

In [0]:
from pyspark.sql.functions import current_timestamp, current_date, col

raw_stream_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", f"{lab3_checkpoint}/schema_inference") 
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("rescuedDataColumn", "_rescued_data") 
    .load(lab3_source_path)
)

In [0]:
bronze_stream_df = (raw_stream_df
    .select("*", col("_metadata.file_name").alias("source_file"))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("load_date", current_date())
)

In [0]:
sq = (bronze_stream_df.writeStream
    .format("delta")
    .option("checkpointLocation", lab3_checkpoint)
    .option("mergeSchema", "true") 
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(table_name)
)

sq.awaitTermination()

In [0]:
display(spark.table(table_name))

In [0]:
mutated_df = spark.sql("SELECT 99.9 as fake_column")

(mutated_df.write
    .mode("append")
    .format("csv")
    .option("header", "true")
    .save(lab3_source_path)
)

In [0]:
display(spark.table(table_name))

In [0]:
display(spark.table(table_name).filter("fake_column is not null"))

![](stream.png)

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {table_name}")
dbutils.fs.rm(lab3_checkpoint, True)